# Specialist-only candidate: Benign/Spoofing refinement of the best submission
**Goal:** keep the best submission (public 0.84585) and change *only* what validation supports.

**Rule for the candidate**
- Start from the best submission's predictions (exact destination port, forward-only Availability model + full model).
- Keep every **Availability** and **Reconnaissance** prediction.
- Keep every **port-5353** prediction (train has only 27 Benign/Spoofing examples there, so nothing is validated).
- For the remaining rows predicted Benign or Spoofing, re-decide with a **binary Benign-vs-Spoofing specialist** (trained only on those two classes).

**Result in one line:** 94 predictions change (all on port 53); validation gain is positive in 9 of 9 paired seed comparisons but is small on this test set.

## 1. Setup and data

In [1]:
import os, warnings
import numpy as np, pandas as pd, lightgbm as lgb
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.model_selection import cross_val_predict
warnings.filterwarnings('ignore')
SEED = 0
np.random.seed(SEED)

# Path: Kaggle input folder if present, otherwise a local folder
for d in ['/kaggle/input/network-intrusion-detection2-ml-challenge', '/kaggle/input', '.', '/home/claude/data']:
    if os.path.exists(os.path.join(d, 'train.csv')):
        DATA = d; break
    for root, _, files in os.walk(d) if os.path.exists(d) else []:
        if 'train.csv' in files: DATA = root; break
print('data dir:', DATA)
tr = pd.read_csv(f'{DATA}/train.csv'); te = pd.read_csv(f'{DATA}/test.csv')
sample = pd.read_csv(f'{DATA}/sample_submission.csv')
CLASSES = ['Availability', 'Benign', 'Reconnaissance', 'Spoofing']
y = tr.Target.map({c: i for i, c in enumerate(CLASSES)}).values
print(tr.shape, te.shape)

data dir: /home/claude/data


(84713, 84) (33161, 83)


## 2. Features (same as the best pipeline)

In [2]:
SVC = {'-':0,'dns':1,'mqtt':2,'ssl':3,'http':4,'ntp':5,'irc':6,'dhcp':7}
COMMON_PORTS = {53,80,123,443,1883,5353,137,21,22,8883,5060}

def make_features(df):
    X = df.drop(columns=['id','Target'], errors='ignore').copy()
    X['proto'] = X.proto.map({'tcp':0,'udp':1,'icmp':2})
    X['svc'] = X.service.map(SVC).fillna(8); X = X.drop(columns='service')
    X['pk_ratio'] = X.fwd_pkts_tot/(X.bwd_pkts_tot+1)
    X['bytes_per_pkt'] = (X['fwd_pkts_payload.tot']+X['bwd_pkts_payload.tot'])/(X.fwd_pkts_tot+X.bwd_pkts_tot)
    X['syn_ack'] = X.flow_SYN_flag_count/(X.flow_ACK_flag_count+1)
    X['dur_log'] = np.log1p(X.flow_duration)
    p = X['id.resp_p'].values
    X['port_bkt'] = np.select([p<1024, p<49152],[0,1],2)          # well-known / registered / ephemeral
    X['port_common'] = np.where(X['id.resp_p'].isin(COMMON_PORTS), X['id.resp_p'], 0)  # exact port only if common
    return X.replace([np.inf,-np.inf], np.nan)

X, Xt = make_features(tr), make_features(te)
print(X.shape)

(84713, 88)


## 3. Validation folds: three ways of holding out unseen traffic
- **S1 cluster:** hold out one KMeans cluster per class.
- **S2 unseen port:** hold out whole destination ports (Availability uses S1 clusters because it has almost only port 21).
- **S3 unseen signature:** hold out whole traffic signatures (protocol, payload, window, packet count).

The same folds are used for the original pipeline and the specialist, so differences are paired.

In [3]:
K = 4
beh = X.drop(columns=[c for c in X.columns if c.startswith('id.') or c.startswith('port_')])
Z = StandardScaler().fit_transform(np.log1p(beh.fillna(0).clip(lower=0)))
fold = np.zeros(len(tr), int)
for k in range(4):
    m = y==k
    fold[m] = KMeans(K, n_init=3, random_state=SEED).fit_predict(Z[m])
print(pd.crosstab(pd.Series(y).map(dict(enumerate(CLASSES))), fold))

def hashfold(v): return (pd.util.hash_array(np.asarray(v).astype('int64')) % K).astype(int)
sigkey = (X.proto.astype(int)*1000003 + X['fwd_pkts_payload.tot'].fillna(0).astype(int)*131
          + X.fwd_init_window_size.fillna(0).astype(int)*7 + X.fwd_pkts_tot.fillna(0).astype(int)).values
nz = y != 0
f2 = fold.copy(); f2[nz] = hashfold(X['id.resp_p'].values[nz].astype(int))
f3 = fold.copy(); f3[nz] = hashfold(sigkey[nz])
SCH = {'S1_cluster': fold, 'S2_unseen_port': f2, 'S3_unseen_signature': f3}

col_0               0    1      2    3
row_0                                 
Availability    31899  280  47800   93
Benign            142  973    404   64
Reconnaissance    139  329    221   46
Spoofing          798  884    470  171


## 4. Model helpers and feature sets

In [4]:
def lgbm(seed, **kw):
    p = dict(n_estimators=300, learning_rate=.05, num_leaves=15, min_child_samples=20,
             subsample=.8, subsample_freq=1, colsample_bytree=.6, verbose=-1, n_jobs=2, random_state=seed)
    p.update(kw); return lgb.LGBMClassifier(**p)

def balanced_sample(idx, seed, cap):
    # cap the huge Availability class, keep all rare-class rows
    rng = np.random.RandomState(seed)
    a = idx[y[idx]==0]; o = idx[y[idx]!=0]
    idx = np.r_[rng.choice(a, min(len(a), cap), replace=False), o]
    w = pd.Series(y[idx]).map(1/pd.Series(y[idx]).value_counts()).values      # inverse class frequency
    return idx, w/w.mean()

def group_cv(cols, seed_base=0):
    pred = np.zeros(len(tr), int)
    for f in range(K):
        idx, w = balanced_sample(np.where(fold!=f)[0], seed_base+f, 6000)
        m = lgbm(seed_base+f).fit(X.iloc[idx][cols], y[idx], sample_weight=w)
        pred[fold==f] = m.predict(X[fold==f][cols])
    return f1_score(y, pred, average='macro'), f1_score(y, pred, average=None)

ALL = [c for c in X.columns if c not in ('id.orig_p','port_bkt','port_common')]
FEATS = {
  'resp_port_exact':  ALL,
  'bucketed_ports':   [c for c in ALL if c!='id.resp_p'] + ['port_bkt'],
  'common_ports':     [c for c in ALL if c!='id.resp_p'] + ['port_common','port_bkt'],
}

In [5]:
fwd_cols = [c for c in ALL if not (c.startswith('bwd') or c.startswith('flow_') or c.startswith('idle') or c.startswith('active')
            or c in ('down_up_ratio','pk_ratio','bytes_per_pkt','syn_ack','payload_bytes_per_second','dur_log'))]
print(len(fwd_cols), 'forward-only features')

def fit_predict(cols, seeds, cap=8000):
    P = np.zeros((len(te), 4))
    for sd in seeds:
        idx, w = balanced_sample(np.arange(len(tr)), sd, cap)
        P += lgbm(sd).fit(X.iloc[idx][cols], y[idx], sample_weight=w).predict_proba(Xt[cols])
    return P/len(seeds)

28 forward-only features


## 5. Reproduce the best submission
Exact-port LightGBM (5 seeds) decides the class; the forward-only model (5 seeds) decides Availability when its probability exceeds 0.5.

In [6]:
P_fwd = fit_predict(fwd_cols, range(5))
P_A   = fit_predict(FEATS['resp_port_exact'], range(5))
pred_A = np.where(P_fwd[:,0] > .5, 0, 1 + P_A[:,1:].argmax(1))
best = pd.DataFrame({'id': te.id, 'Target': np.array(CLASSES)[pred_A]})
print(best.Target.value_counts().to_dict())
ref = '/home/claude/work/submission_final.csv'
if os.path.exists(ref):
    print('identical to the submitted best file:', bool((pd.read_csv(ref).Target.values == best.Target.values).all()))

{'Benign': 10475, 'Availability': 10391, 'Reconnaissance': 7514, 'Spoofing': 4781}
identical to the submitted best file: True


## 6. The Benign-vs-Spoofing specialist
Trained only on Benign and Spoofing rows, balanced 50/50, shallower trees (`num_leaves=7`, `min_child_samples=40`), common-port features, 7 seeds averaged. Threshold 0.5 (not tuned, because tuning on held-out profiles would just overfit them).

In [7]:
cols_S = FEATS['common_ports']
bs = np.where((y==1)|(y==3))[0]; yb = (y[bs]==3).astype(int)
def wbin(b): return pd.Series(b).map(1/pd.Series(b).value_counts()).values
w = wbin(yb); w = w/w.mean()
pb = np.zeros(len(te))
for sd in range(7):
    pb += lgbm(sd, num_leaves=7, min_child_samples=40).fit(X.iloc[bs][cols_S], yb, sample_weight=w).predict_proba(Xt[cols_S])[:,1]/7

eligible = best.Target.isin(['Benign','Spoofing']).values & (te['id.resp_p'].values != 5353)
cand = best.copy()
cand.loc[eligible, 'Target'] = np.where(pb[eligible] > .5, 'Spoofing', 'Benign')
print('eligible rows:', eligible.sum(), '| changed:', (cand.Target != best.Target).sum())
cand.to_csv('specialist_only_candidate.csv', index=False)

eligible rows: 13844 | changed: 94


## 7. Checks on the candidate

In [8]:
ss = pd.read_csv(f'{DATA}/sample_submission.csv')
d = cand.Target != best.Target
print('rows', len(cand), '| columns', list(cand.columns), '| missing', int(cand.isna().sum().sum()), '| duplicate ids', int(cand.id.duplicated().sum()))
print('valid labels:', set(cand.Target) <= set(CLASSES), '| ids match sample_submission in order:', bool((cand.id == ss.id).all()))
print('Availability preserved:', bool(((best.Target=='Availability') == (cand.Target=='Availability')).all()),
      '| Reconnaissance preserved:', bool(((best.Target=='Reconnaissance') == (cand.Target=='Reconnaissance')).all()))
m5353 = (te['id.resp_p'] == 5353).values
print('port-5353 rows:', m5353.sum(), '| changed among them:', int((best.Target[m5353] != cand.Target[m5353]).sum()))
print('\nclass transitions (original -> candidate):'); print(pd.crosstab(best.Target[d], cand.Target[d]))
print('\nchanged rows by port:', te['id.resp_p'][d].value_counts().to_dict())
print('\nclass distribution'); print(pd.DataFrame({'original': best.Target.value_counts(), 'candidate': cand.Target.value_counts()}))

rows 33161 | columns ['id', 'Target'] | missing 0 | duplicate ids 0
valid labels: True | ids match sample_submission in order: True
Availability preserved: True | Reconnaissance preserved: True
port-5353 rows: 2009 | changed among them: 0

class transitions (original -> candidate):
Target    Benign  Spoofing
Target                    
Benign         0        36
Spoofing      58         0

changed rows by port: {53.0: 94}

class distribution
                original  candidate
Target                             
Benign             10475      10497
Availability       10391      10391
Reconnaissance      7514       7514
Spoofing            4781       4759


## 8. Validation: original pipeline vs original + specialist (identical folds)
Per fold, a model is trained on the other folds and scored on the held-out one. "A" is the original pipeline; "A_spec" applies the specialist to rows A calls Benign/Spoofing (excluding port 5353). Three seeds per scheme.

In [9]:
not5353 = (X['id.resp_p'].values != 5353)
def run(fv, seed):
    out = {k: np.zeros(len(tr), int) for k in ['A','A_spec']}
    for f in range(K):
        tri = np.where(fv != f)[0]; va = fv == f; s = seed*10 + f
        idx, wt = balanced_sample(tri, s, 6000); Xf = X.iloc[idx]
        mf = lgbm(s).fit(Xf[ALL], y[idx], sample_weight=wt)
        ma = lgbm(s).fit(Xf[fwd_cols], y[idx], sample_weight=wt)
        bi = idx[(y[idx]==1)|(y[idx]==3)]; bb = (y[bi]==3).astype(int)
        mb = lgbm(s, num_leaves=7, min_child_samples=40).fit(X.iloc[bi][cols_S], bb, sample_weight=wbin(bb))
        Xv = X[va]; pf = mf.predict_proba(Xv[ALL]); avail = ma.predict_proba(Xv[fwd_cols])[:,0] > .5
        base = np.where(avail, 0, 1 + pf[:,1:].argmax(1)); out['A'][va] = base
        p_s = mb.predict_proba(Xv[cols_S])[:,1]; q = base.copy()
        bm = ((base==1)|(base==3)) & not5353[va]; q[bm] = np.where(p_s[bm] > .5, 3, 1); out['A_spec'][va] = q
    return out

pooled = {}; foldrows = []
for sn, fv in SCH.items():
    for seed in range(3):
        o = run(fv, seed)
        for k in o: pooled[(sn,k,seed)] = (f1_score(y, o[k], average='macro'), f1_score(y, o[k], average=None))
        for f in range(4):
            m = fv == f; labs = sorted(set(y[m]))
            foldrows.append(dict(scheme=sn, seed=seed, fold=f,
                A=f1_score(y[m], o['A'][m], labels=labs, average='macro'),
                A_spec=f1_score(y[m], o['A_spec'][m], labels=labs, average='macro')))
    print(sn, 'done', flush=True)

S1_cluster done


S2_unseen_port done


S3_unseen_signature done


In [10]:
rows = []
for sn in SCH:
    a = np.array([pooled[(sn,'A',s)][0] for s in range(3)]); b = np.array([pooled[(sn,'A_spec',s)][0] for s in range(3)])
    rows.append(dict(scheme=sn, original=a.mean().round(3), specialist=b.mean().round(3), difference=(b-a).mean().round(3),
                     seed_diffs=np.round(b-a, 4).tolist(), positive_in_all_seeds=bool(((b-a) > 0).all())))
print(pd.DataFrame(rows).to_string(index=False))
print()
for sn in SCH:
    for k in ['A','A_spec']:
        pc = np.mean([pooled[(sn,k,s)][1] for s in range(3)], 0).round(3)
        print(f'{sn:20s} {k:7s} per-class F1 [Avail, Benign, Recon, Spoof]: {pc}')
fr = pd.DataFrame(foldrows); fr['delta'] = fr.A_spec - fr.A
g = fr.groupby(['scheme','fold']).agg(original=('A','mean'), specialist=('A_spec','mean'), delta=('delta','mean'), worst_seed=('delta','min'), best_seed=('delta','max')).round(3)
print('\nPer-fold (mean over seeds):'); print(g.to_string())
print('\nfolds improved by >0.01:', int((fr.groupby(['scheme','fold']).delta.mean() > .01).sum()), 'of 12 | folds never worse in any seed:', int((fr.groupby(['scheme','fold']).delta.min() >= -1e-9).sum()), 'of 12 | worst single seed-fold delta:', round(fr.delta.min(), 4))

             scheme  original  specialist  difference               seed_diffs  positive_in_all_seeds
         S1_cluster     0.641       0.653       0.012   [0.01, 0.0102, 0.0171]                   True
     S2_unseen_port     0.718       0.805       0.088 [0.1157, 0.0576, 0.0905]                   True
S3_unseen_signature     0.791       0.792       0.001 [0.0011, 0.0012, 0.0012]                   True

S1_cluster           A       per-class F1 [Avail, Benign, Recon, Spoof]: [0.996 0.616 0.342 0.609]
S1_cluster           A_spec  per-class F1 [Avail, Benign, Recon, Spoof]: [0.996 0.634 0.342 0.641]
S2_unseen_port       A       per-class F1 [Avail, Benign, Recon, Spoof]: [0.996 0.591 0.823 0.46 ]
S2_unseen_port       A_spec  per-class F1 [Avail, Benign, Recon, Spoof]: [0.996 0.73  0.823 0.673]
S3_unseen_signature  A       per-class F1 [Avail, Benign, Recon, Spoof]: [0.997 0.937 0.455 0.776]
S3_unseen_signature  A_spec  per-class F1 [Avail, Benign, Recon, Spoof]: [0.997 0.937 0.455 0.78

## 9. Interpretation and limits
- **Direction is consistent:** the specialist is better in all 9 paired seed comparisons (3 schemes x 3 seeds). The gain is large on unseen ports (S2), small on cluster hold-out (S1) and negligible on unseen signatures (S3).
- **The gain is concentrated:** only about four of twelve folds improve by more than 0.01; the rest are within +/-0.005 and a few are slightly negative.
- **Only Benign/Spoofing change:** Availability and Reconnaissance are untouched by construction.
- **Small real-world effect:** the candidate changes just 94 of 33,161 test rows (all port 53, nearly balanced 36 vs 58), so a leaderboard movement larger than about 0.002 is not expected.
- **Calibration:** the specialist's probabilities are overconfident under shift and it under-predicts Spoofing in held-out folds, so the 0.5 threshold is not guaranteed optimal; it was left untuned on purpose.
- **Validation is a proxy** built by me; it cannot confirm the organizers' private profile split.
- **Port 5353 (about 2,000 flows)** is deliberately left as the best file labels it. Train cannot validate either label, and the public score cannot either, since these flows appear to lie in the private split.